# 🤖 Notebook 03 — Trenowanie Modeli
## Model Training: Random Forest + LSTM

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
plt.style.use('dark_background')
Path('../models').mkdir(exist_ok=True)
Path('../reports').mkdir(exist_ok=True)
SEED = 42
WINDOW_SIZE = 30
STEP = 15
print('Setup complete.')

## 1. Load / Generate Data

In [ ]:
landmark_dir = Path('../data/landmarks')
if landmark_dir.exists() and any(landmark_dir.rglob('*.csv')):
    from src.data.loader import LandmarkLoader
    df = LandmarkLoader().load_directory(str(landmark_dir))
    print(f'Loaded {len(df):,} rows.')
else:
    print('No CSV data found — generating synthetic data (demo mode).')
    from src.data.generator import SyntheticGenerator
    df = SyntheticGenerator(seed=SEED).generate(n_normal=400, n_anomaly=200)
    print(f'Generated {len(df):,} synthetic rows.')
print(f'Class distribution: {dict(df.groupby("label").size())}')

## 2. Feature Engineering

In [ ]:
from src.data.features import FeatureExtractor
fe = FeatureExtractor()
X_frame, y_frame = fe.frame_features(df, label_col='label')
print(f'Frame features: {X_frame.shape}')
X_seq, y_seq = fe.window_features(df, label_col='label', window_size=WINDOW_SIZE, step=STEP)
print(f'Window features: {X_seq.shape}')
X_tr, X_te, y_tr, y_te = train_test_split(X_frame.values, y_frame.values, test_size=0.2, stratify=y_frame.values, random_state=SEED)
X_seq_tr, X_seq_te, y_seq_tr, y_seq_te = train_test_split(X_seq, y_seq, test_size=0.2, stratify=y_seq, random_state=SEED)
print(f'RF splits: train={len(y_tr)}, test={len(y_te)}')
print(f'LSTM splits: train={len(y_seq_tr)}, test={len(y_seq_te)}')

## 3. Train Random Forest

In [ ]:
from src.models.random_forest import FallRandomForest
from sklearn.metrics import classification_report
rf = FallRandomForest(n_estimators=200, random_state=SEED)
rf.fit(X_tr, y_tr)
y_pred_rf = rf.predict(X_te)
y_proba_rf = rf.predict_proba(X_te)
print(classification_report(y_te, y_pred_rf, target_names=['Normal', 'Anomaly']))
rf.save('../models/random_forest.pkl')
print('Saved: models/random_forest.pkl')

### Feature Importances

In [ ]:
importances = rf.feature_importances
if importances is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor('#0d0d1a')
    ax.set_facecolor('#1a1a2e')
    top10 = importances.head(10)
    ax.barh(top10.index[::-1], top10.values[::-1], color='#2196F3')
    ax.set_title('Top 10 Feature Importances — Random Forest', color='white')
    ax.set_xlabel('Importance', color='#aaa')
    ax.tick_params(colors='#aaa')
    plt.tight_layout()
    plt.savefig('../reports/feature_importances.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
    plt.show()

## 4. Train LSTM

In [ ]:
from src.models.lstm import FallLSTM
n_features = X_seq_tr.shape[2]
lstm = FallLSTM(window_size=WINDOW_SIZE, n_features=n_features, class_weight={0: 1.0, 1: 3.0})
lstm.summary()
lstm.fit(X_seq_tr, y_seq_tr, X_val=X_seq_te, y_val=y_seq_te, epochs=50, patience=10)
y_pred_lstm = lstm.predict(X_seq_te)
y_proba_lstm = lstm.predict_proba(X_seq_te)
print(classification_report(y_seq_te, y_pred_lstm, target_names=['Normal', 'Anomaly']))
lstm.save('../models/lstm_model.keras')
print('Saved: models/lstm_model.keras')

### Training History

In [ ]:
if lstm.history:
    from src.evaluation.metrics import Evaluator
    Evaluator(save_dir='../reports', show_plots=True).training_history_plot(lstm.history)

## 5. Model Comparison Summary

In [ ]:
from src.evaluation.metrics import Evaluator
min_len = min(len(y_te), len(y_seq_te))
ev = Evaluator(save_dir='../reports', show_plots=True)
metrics = ev.summary_table(
    y_true=y_te[:min_len],
    y_pred_rf=y_pred_rf[:min_len],
    y_pred_lstm=y_pred_lstm[:min_len],
    y_proba_rf=y_proba_rf[:min_len],
    y_proba_lstm=y_proba_lstm[:min_len],
)
print('Model Comparison:')
metrics